In [5]:

import os

BASE_DIR = "/content/scholarshipscan"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("ScholarshipScan folders created successfully!")
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


ScholarshipScan folders created successfully!
Raw folder: /content/scholarshipscan/data/raw
Processed folder: /content/scholarshipscan/data/processed


In [6]:

from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    destination = os.path.join(RAW_DIR, filename)
    shutil.move(filename, destination)

print("\nUploaded files:")

for filename in os.listdir(RAW_DIR):
    print(filename)


Saving mis.pdf to mis.pdf
Saving mtcp.pdf to mtcp.pdf
Saving mext 2.pdf to mext 2.pdf

Uploaded files:
mext 2.pdf
mis.pdf
mtcp.pdf
mybrain.pdf


In [7]:

from pypdf import PdfReader

for filename in os.listdir(RAW_DIR):
    if filename.lower().endswith(".pdf"):
        path = os.path.join(RAW_DIR, filename)

        reader = PdfReader(path)

        print("\nFile:", filename)
        print("Pages:", len(reader.pages))

        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""

        print("Extracted characters:", len(text))
        print("Sample text:")
        print(text[:500])



File: mext 2.pdf
Pages: 2
Extracted characters: 6397
Sample text:
JAPANESE GOVERNMENT (MONBUKAGAKUSHO: MEXT) SCHOLARSHIP  
FOR ACADEMIC YEAR 2027 
 
UNDERGRADUATE STUDENTS 
 
The Monbukagakusho:MEXT (Ministry of Education, Culture, Sports, Science and Technology, Government of 
Japan) offers scholarships to Malaysian students who wish to study at Japanese universities as undergraduate 
students under the Japanese Government (Monbukagakusho) Scholarship Program for the academic year 2027.  
Applicants must be Malaysian citizens and meet the qualifications state

File: mis.pdf
Pages: 7
Extracted characters: 9006
Sample text:
1 
 
GUIDELINES FOR THE APPLICATION OF  
MALAYSIA INTERNATIONAL SCHOLARSHIP (MIS) 2026 
Malaysia International Scholarship (MIS) 2026 
WHO CAN 
APPLY 
 Graduates from the recipient countries of MIS 2026 who:  
1. are citizens from the recipient countries* of MIS 2026;  
2. intend to pursue full-time postgraduate studies 
(Master’s/PhD) in Malaysia; or 
3. are curren

In [8]:

from pathlib import Path
import shutil
from google.colab import files

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

for folder in [RAW_DIR, PROCESSED_DIR, PIPE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

expected_files = ["mis.pdf", "mtcp.pdf", "mybrain.pdf"]

old_dir = Path("/content/scholarshipscan/data/raw")

for filename in expected_files:
    target = RAW_DIR / filename

    if target.exists():
        continue

    old_file = old_dir / filename

    if old_file.exists():
        shutil.copy2(old_file, target)

missing = [
    name for name in expected_files
    if not (RAW_DIR / name).exists()
]

if missing:
    print("Please upload:", missing)
    uploaded = files.upload()

    for filename, content in uploaded.items():
        if filename in expected_files:
            (RAW_DIR / filename).write_bytes(content)

print("\nPROJECT FILE CHECK")

for filename in expected_files:
    path = RAW_DIR / filename
    print(filename, "FOUND" if path.exists() else "MISSING")

assert all((RAW_DIR / f).exists() for f in expected_files), \
    "Please upload all three PDF documents."

print("\nCell 1 completed successfully.")



PROJECT FILE CHECK
mis.pdf FOUND
mtcp.pdf FOUND
mybrain.pdf FOUND

Cell 1 completed successfully.


In [9]:

!pip -q install pypdf

from pypdf import PdfReader

inspection_results = []

for filename in expected_files:
    path = RAW_DIR / filename
    reader = PdfReader(str(path))

    extracted_text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    result = {
        "filename": filename,
        "pages": len(reader.pages),
        "characters": len(extracted_text),
        "readable": bool(extracted_text.strip())
    }

    inspection_results.append(result)

    print("\nFile:", filename)
    print("Pages:", result["pages"])
    print("Characters:", result["characters"])
    print("Readable:", result["readable"])
    print("Sample:")
    print(extracted_text[:300])

assert all(r["readable"] for r in inspection_results)

print("\nAll PDF documents are readable.")



File: mis.pdf
Pages: 7
Characters: 9012
Readable: True
Sample:
1 
 
GUIDELINES FOR THE APPLICATION OF  
MALAYSIA INTERNATIONAL SCHOLARSHIP (MIS) 2026 
Malaysia International Scholarship (MIS) 2026 
WHO CAN 
APPLY 
 Graduates from the recipient countries of MIS 2026 who:  
1. are citizens from the recipient countries* of MIS 2026;  
2. intend to pursue full-time

File: mtcp.pdf
Pages: 4
Characters: 7462
Readable: True
Sample:
 
Page 1 of 4 
 
GUIDELINES 
FOR THE MALAYSIAN TECHNICAL COOPERATION PROGRAMME 
(MTCP) SCHOLARSHIP 2026                                                                                                                                                                                                     

File: mybrain.pdf
Pages: 10
Characters: 10568
Readable: True
Sample:
SOALAN LAZIM BIASISWA MYBRAINSC TAHUN 2026 
 
 
SOALAN JAWAPAN 
 
1. 
 
Bila tarikh permohonan 
biasiswa ini dibuka? 
 
Permohonan ini dibuka mulai pada 3 Mac 2026, jam 
12.00 tengah hari sehingga 2 

In [10]:

import re
import json
from pypdf import PdfReader

CHUNK_SIZE = 500

def clean_text(text):
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def fixed_chunking(text, size=500):
    return [
        text[i:i + size]
        for i in range(0, len(text), size)
        if text[i:i + size].strip()
    ]


def paragraph_chunking(text, size=500):
    chunks = []
    current = ""

    paragraphs = re.split(r"\n\s*\n", text)

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        if len(paragraph) > size:
            if current:
                chunks.append(current)
                current = ""

            chunks.extend(
                paragraph[i:i + size]
                for i in range(0, len(paragraph), size)
            )

        elif len(current) + len(paragraph) + (
            2 if current else 0
        ) <= size:

            current += (
                "\n\n" if current else ""
            ) + paragraph

        else:
            if current:
                chunks.append(current)

            current = paragraph

    if current:
        chunks.append(current)

    return chunks


fixed_records = []
paragraph_records = []

for filename in expected_files:
    reader = PdfReader(str(RAW_DIR / filename))

    for page_number, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")

        if not text:
            continue

        fixed_chunks = fixed_chunking(text, CHUNK_SIZE)
        paragraph_chunks = paragraph_chunking(text, CHUNK_SIZE)

        for i, chunk in enumerate(fixed_chunks, start=1):
            fixed_records.append({
                "source": filename,
                "page": page_number,
                "chunk_id": f"{filename}_p{page_number}_f{i}",
                "text": chunk
            })

        for i, chunk in enumerate(paragraph_chunks, start=1):
            paragraph_records.append({
                "source": filename,
                "page": page_number,
                "chunk_id": f"{filename}_p{page_number}_p{i}",
                "text": chunk
            })


# Save both comparison outputs
comparison_dir = PROCESSED_DIR / "chunking_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)

for name, records in [
    ("fixed_chunks.jsonl", fixed_records),
    ("paragraph_chunks.jsonl", paragraph_records)
]:
    with open(comparison_dir / name, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def chunk_statistics(records):
    lengths = [len(r["text"]) for r in records]

    return {
        "total_chunks": len(records),
        "average_characters": round(
            sum(lengths) / len(lengths), 2
        ) if lengths else 0,
        "maximum_characters": max(lengths, default=0),
        "minimum_characters": min(lengths, default=0)
    }


print("FIXED-SIZE CHUNKING")
print(chunk_statistics(fixed_records))

print("\nPARAGRAPH-AWARE CHUNKING")
print(chunk_statistics(paragraph_records))

print("\nComparison files saved:", comparison_dir)


FIXED-SIZE CHUNKING
{'total_chunks': 63, 'average_characters': 421.84, 'maximum_characters': 500, 'minimum_characters': 60}

PARAGRAPH-AWARE CHUNKING
{'total_chunks': 84, 'average_characters': 312.67, 'maximum_characters': 500, 'minimum_characters': 1}

Comparison files saved: /content/week3_project/data/processed/chunking_comparison


In [11]:

import json

selected_records = paragraph_records

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"

with open(prepared_path, "w", encoding="utf-8") as f:
    for record in selected_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Basic output contract
assert prepared_path.is_file(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."

# Reload and inspect representative records
with open(prepared_path, "r", encoding="utf-8") as f:
    exported = [json.loads(line) for line in f if line.strip()]

assert len(exported) == len(selected_records), \
    "Exported record count mismatch."

assert all(
    len(record["text"]) <= CHUNK_SIZE
    for record in exported
), "A chunk exceeds the configured size."

print("Output:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Prepared chunks:", len(exported))
print("Validation: PASS")

print("\nRepresentative prepared records:")

for record in exported[:3]:
    print("\nSource:", record["source"])
    print("Page:", record["page"])
    print("Chunk ID:", record["chunk_id"])
    print("Text:", record["text"][:250])


Output: /content/week3_project/data/processed/prepared_text.jsonl
File size: 33781 bytes
Prepared chunks: 84
Validation: PASS

Representative prepared records:

Source: mis.pdf
Page: 1
Chunk ID: mis.pdf_p1_p1
Text: 1

Source: mis.pdf
Page: 1
Chunk ID: mis.pdf_p1_p2
Text: GUIDELINES FOR THE APPLICATION OF 
MALAYSIA INTERNATIONAL SCHOLARSHIP (MIS) 2026 
Malaysia International Scholarship (MIS) 2026 
WHO CAN 
APPLY 
 Graduates from the recipient countries of MIS 2026 who: 
1. are citizens from the recipient countries* o

Source: mis.pdf
Page: 1
Chunk ID: mis.pdf_p1_p3
Text: inimum of two (2) 
semesters remaining before the end of the study period.


In [12]:

import hashlib
import json
from datetime import datetime, timezone

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


lineage_records = []

for i, filename in enumerate(expected_files, start=1):
    source = RAW_DIR / filename

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": filename,
        "source_sha256": sha256_file(source),
        "prepared_output": str(
            prepared_path.relative_to(BASE_DIR)
        ),
        "prepared_sha256": sha256_file(prepared_path),
        "processing_method": (
            "PDF text extraction; whitespace cleaning; "
            "paragraph-aware chunking; max 500 characters"
        ),
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })


manifest = {
    "project": "ScholarshipScan",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "records": lineage_records
}

manifest_path = BASE_DIR / "data" / "week3_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False)

print("COMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(lineage_records))
print("Saved to:", manifest_path)


COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 3
Saved to: /content/week3_project/data/week3_manifest.json


In [13]:

with open(manifest_path, "r", encoding="utf-8") as f:
    saved_manifest = json.load(f)

saved_records = saved_manifest["records"]

assert len(saved_records) == len(expected_files)

for record in saved_records:
    source = RAW_DIR / record["source_filename"]

    assert source.is_file()
    assert prepared_path.is_file()

    assert sha256_file(source) == record["source_sha256"]
    assert sha256_file(prepared_path) == record["prepared_sha256"]

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)


COMMON LINEAGE VALIDATION: PASS
Verified records: 3
Manifest: /content/week3_project/data/week3_manifest.json


In [14]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(source)
    else:
        if not source.is_file():
            raise FileNotFoundError(source)

    return source


def execute_pipeline(
    kind, source, inspect_fn, prepare_fn, configuration
):
    source = Path(source)
    inputs = [source]

    # INSPECT
    inspection = inspect_fn(inputs, configuration)

    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE
    outputs, mappings, method = prepare_fn(
        inputs, output_dir, configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE
    if not outputs:
        raise ValueError("No prepared outputs produced.")

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(output)

        if output.stat().st_size == 0:
            raise ValueError("Prepared output is empty.")

    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": "Not verified"
    }

    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(record, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


def register_and_run(
    kind, source, inspect_fn, prepare_fn, configuration
):
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }

    return first


print("Cell 7 ready.")
print("Run the selected Cell 8A pathway next.")


Cell 7 ready.
Run the selected Cell 8A pathway next.


In [15]:

import re
import json
from pypdf import PdfReader

# Selected Part B configuration
CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}


def clean_document_text(text):
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_paragraphs(text, size):
    chunks = []
    current = ""

    paragraphs = re.split(r"\n\s*\n", text)

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        if len(paragraph) > size:
            if current:
                chunks.append(current)
                current = ""

            chunks.extend(
                paragraph[i:i + size]
                for i in range(0, len(paragraph), size)
            )

        elif len(current) + len(paragraph) + (
            2 if current else 0
        ) <= size:

            current += (
                "\n\n" if current else ""
            ) + paragraph

        else:
            if current:
                chunks.append(current)
            current = paragraph

    if current:
        chunks.append(current)

    return chunks


# INSPECT DOCUMENTS
def inspect_documents(inputs, cfg):
    src = inputs[0]
    reader = PdfReader(str(src))

    pages = [
        page.extract_text() or ""
        for page in reader.pages
    ]

    text = "\n".join(pages)

    if not text.strip():
        raise ValueError(
            "No extractable text found. PDF may require OCR."
        )

    return {
        "source": src.name,
        "pages": len(pages),
        "characters": len(text)
    }


# PREPARE DOCUMENTS
def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    reader = PdfReader(str(src))

    records = []
    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive.")

    for page_number, page in enumerate(reader.pages, start=1):
        text = clean_document_text(
            page.extract_text() or ""
        )

        if not text:
            continue

        if method == "paragraph-aware":
            chunks = split_paragraphs(text, size)

        elif method == "fixed":
            chunks = [
                text[i:i + size]
                for i in range(0, len(text), size)
                if text[i:i + size].strip()
            ]

        else:
            raise ValueError(
                "Choose fixed or paragraph-aware."
            )

        for i, chunk in enumerate(chunks, start=1):
            records.append({
                "id": len(records) + 1,
                "source": src.name,
                "text": chunk,
                "page": page_number
            })

    if not records:
        raise ValueError("No text chunks produced.")

    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    mapping = [{
        "source": str(src),
        "output": str(target)
    }]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method


# RUN ALL THREE PDF DOCUMENTS
run_results = {}

for filename in expected_files:
    source = resolve_source(filename)

    pipeline_name = (
        "documents_" + source.stem
    )

    run_results[pipeline_name] = register_and_run(
        pipeline_name,
        source,
        inspect_documents,
        prepare_documents,
        CONFIG_A
    )

print("\nCell 8A completed successfully.")
print("Processed documents:", len(run_results))


SUCCESS: documents_mis | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mis/pipeline_manifest.json
SUCCESS: documents_mtcp | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mtcp/pipeline_manifest.json
SUCCESS: documents_mybrain | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mybrain/pipeline_manifest.json

Cell 8A completed successfully.
Processed documents: 3


In [16]:

assert PIPELINE_RUNS, (
    "No pipeline runs registered. Run Cell 8A first."
)

print("--- Repeatability Verification ---")

for kind, entry in PIPELINE_RUNS.items():

    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Verify source fingerprint
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Verify prepared output fingerprint
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} - source fingerprints, "
        "prepared outputs and mappings match"
    )

print("\nAll selected pipelines passed repeatability smoke checks.")


--- Repeatability Verification ---
SUCCESS: documents_mis | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mis/pipeline_manifest.json
PASS: documents_mis - source fingerprints, prepared outputs and mappings match
SUCCESS: documents_mtcp | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mtcp/pipeline_manifest.json
PASS: documents_mtcp - source fingerprints, prepared outputs and mappings match
SUCCESS: documents_mybrain | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mybrain/pipeline_manifest.json
PASS: documents_mybrain - source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [17]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/week3_project",
    "zip",
    "/content",
    "week3_project"
)

files.download("/content/week3_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>